# What does VERDANT bring for PANGAEA names?

1. Write the PANGAEA parameter names in the list below.
2. Run all cells (the first run takes a few minutes, VERDANT waits for web services).
3. You get one table: for each name, what VERDANT links in each vocabulary (ENVO, ChEBI, Wikidata, GBIF, NCBITaxon, DBpedia). It is also saved as Excel.

VERDANT is not changed. The names are only written into its input file.


In [12]:
# ---- 1. YOUR INPUT: PANGAEA parameter names ----
names = [
    "Mass",
    "Tissue type",
    "Lactate, per fresh mass",
    "Taurine, per fresh mass",
    "Tank number",
    "Humus",
    "Diatoms, upwelling",
    "Bulimina marginata",
    "Aconitate, per fresh mass",
    "Carnitine, per fresh mass",
    "gamma-Aminobutyric acid, per fresh mass"
]
RUN_VERDANT = True            # False = do not run again, show the results of the last run
EXTRA_ONTOLOGIES = ["chebi"]  # other ontologies to search with VERDANT's own matcher (any OLS4 id)

# ---- take the names from the ground-truth CSV instead of the list above (then they are compared with PANGAEA's own terms) ----
USE_DATASET_CSV = True        # True = use all names of the CSV below (59 names, first run takes 30+ minutes). False = use the list above
DATASET_CSV = "dataset_980734_parameters_method_terms.csv"     # in the experiments folder

## 2. Run VERDANT


In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
ROOT = HERE if (HERE / "pipeline").exists() else HERE.parent       # the repo folder
DATA_DIR = ROOT / "data_pangaea_input"                              # VERDANT reads and writes here
sys.path.insert(0, str(ROOT))

# the ground truth: the CSV holds PANGAEA's own terms for each parameter of the dataset
ground_truth = {}                                   # {name: [PANGAEA terms]}
if USE_DATASET_CSV:
    gt = pd.read_csv(ROOT / "experiments" / DATASET_CSV).fillna("")
    names = list(dict.fromkeys(gt["name"]))
    for n, terms_text in zip(gt["name"], gt["terms"]):
        ground_truth.setdefault(n, []).extend(x.strip() for x in terms_text.split(";") if x.strip())
    print(len(names), "names taken from", DATASET_CSV)

# each name is written twice into VERDANT's input file: as "habitat" (route to ENVO) and as "species" (route to GBIF, Wikidata, NCBITaxon, DBpedia)
habitat = [{"id": f"ds_{i:03d}", "label": n, "category": "habitat", "definition": "PANGAEA parameter",
            "codes": [], "scientific_name": None, "common_names": [], "source_url": ""} for i, n in enumerate(names)]
species = [{**t, "id": t["id"] + "_all", "category": "species", "scientific_name": t["label"]} for t in habitat]

if RUN_VERDANT:
    DATA_DIR.mkdir(exist_ok=True)
    (DATA_DIR / "terms.json").write_text(json.dumps(habitat + species, indent=2, ensure_ascii=False), encoding="utf-8")
    cmd = [sys.executable, "-m", "pipeline.main", "--steps", "3,4", "--data-dir", str(DATA_DIR)]
    if subprocess.run(cmd, cwd=ROOT).returncode != 0:
        raise RuntimeError("VERDANT reported an error (see the messages above).")

mappings = json.loads((DATA_DIR / "mappings.json").read_text(encoding="utf-8"))
envo_candidates = json.loads((DATA_DIR / "envo_candidates.json").read_text(encoding="utf-8"))
print(len(mappings), "links found")

## 3. Search the extra ontologies (ChEBI) with VERDANT's own matcher


In [ ]:
import requests
from geoapp import ontolink            # the matcher behind the viewer's ontology manager

session = requests.Session()
session.headers.update({"Accept": "application/json"})
extra = {}                              # {ontology: {name: [hits]}}
for onto in EXTRA_ONTOLOGIES:
    extra[onto] = {n: ontolink.ols4_search(n, onto, session) for n in names}
    print(onto, ":", sum(1 for v in extra[onto].values() if v), "of", len(names), "names have a hit")

## 3b. All candidates

VERDANT itself collects candidates only for ENVO. For the other vocabularies it keeps just the final link. To see the candidates there too, this cell asks the same web services (with VERDANT's own search helpers): OLS4 across **all** ontologies except ENVO, Wikidata and GBIF. These candidates are not VERDANT's own output and nobody has checked them. Together with the ENVO candidates they make the column "All candidates".


In [ ]:

from pipeline import linker            # VERDANT's own search helpers and web addresses
linker.CACHE_DIR = DATA_DIR / "cache" / "linker"

def candidates_from_other_vocabularies(name):
    """Candidates for one name from OLS4 (all ontologies except ENVO), Wikidata and GBIF."""
    out, seen = [], set()
    for query in dict.fromkeys([name] + linker._key_words(name)):          # the name and its key words, as VERDANT does for ENVO
        res = linker._get_json(linker.OLS4_SEARCH, params={"q": query, "type": "class", "rows": 10,
                                                           "fieldList": "iri,label,ontology_name"})
        for d in (res or {}).get("response", {}).get("docs", []):
            if d.get("ontology_name") != "envo" and d["iri"] not in seen and len(seen) < 30:
                seen.add(d["iri"])
                out.append({"vocabulary": f"OLS4:{d.get('ontology_name', '?')}", "label": d.get("label", ""), "uri": d["iri"]})
    res = linker._get_json(linker.WIKIDATA_API, params={"action": "wbsearchentities", "search": name,
                                                        "language": "en", "limit": 5, "format": "json"})
    for d in (res or {}).get("search", []):
        out.append({"vocabulary": "Wikidata", "label": d.get("label", ""), "uri": f"https://www.wikidata.org/entity/{d.get('id', '')}"})
    res = linker._get_json("https://api.gbif.org/v1/species/suggest", params={"q": name, "limit": 5})
    for d in (res or []):
        out.append({"vocabulary": "GBIF", "label": d.get("canonicalName", d.get("scientificName", "")),
                    "uri": f"https://www.gbif.org/species/{d.get('key', '')}"})
    return out

all_candidates = {}                     # {name: [every candidate: vocabulary, label, uri]}
for t in habitat:
    envo = [{"vocabulary": "ENVO", "label": c["label"], "uri": c["uri"]} for c in envo_candidates.get(t["id"], [])]
    all_candidates[t["label"]] = envo + candidates_from_other_vocabularies(t["label"])
print(sum(len(v) for v in all_candidates.values()), "candidates in total for", len(names), "names")


## 4. The result: one row per name, one column per vocabulary


In [ ]:
VOCAB = {"ols4_envo": "ENVO", "wikidata": "Wikidata", "gbif": "GBIF", "ols4_ncbitaxon": "NCBITaxon", "dbpedia": "DBpedia"}

def links(source_id, source_ontology):
    ms = [m for m in mappings if m["source_id"] in (source_id, source_id + "_all") and m["source_ontology"] == source_ontology]
    return " ; ".join(f"{m['target_label']} ({m['match_type'].replace('skos:', '')})" for m in sorted(ms, key=lambda m: -m["confidence"]))

rows = []
for t in habitat:
    row = {"PANGAEA name": t["label"]}
    for key, title in VOCAB.items():
        row[title] = links(t["id"], key)
    for onto, hits in extra.items():
        best = hits.get(t["label"], [])
        row[onto.upper()] = f"{best[0].label} ({best[0].match_type}, {best[0].confidence:.2f})" if best else ""
    row["All candidates"] = " ; ".join(f"{c['label']} [{c['vocabulary']}]" for c in all_candidates[t["label"]])
    rows.append(row)
result = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
result

## 5. Compare with the ground truth

For each name: are PANGAEA's own terms (the ground truth from the CSV) among what VERDANT found? They are compared by name, ignoring upper and lower case.

- **found as a final link:** a PANGAEA term is among the links VERDANT kept (any vocabulary, plus the ChEBI search above).
- **found among candidates only:** a PANGAEA term is among the candidates (stage 1) but was not kept.
- **not found:** no PANGAEA term appears at all.
- **no ground-truth term:** PANGAEA has no term for this parameter, so there is nothing to compare.

Keep in mind that the ground truth is weak: the terms are plain words, some are wrong, and generic words such as `treatment` or `change` match easily. Expert checks the rest.


In [ ]:

import re

def norm(x):
    return re.sub(r"\s+", " ", str(x)).strip().lower()

if not ground_truth:
    comparison = pd.DataFrame()
    print("No ground truth: set USE_DATASET_CSV = True at the top to take the names and PANGAEA's terms from the CSV.")
else:
    rows = []
    for t in habitat:
        gt_terms = ground_truth.get(t["label"], [])
        gset = {norm(x) for x in gt_terms}
        final = [(m["target_label"], VOCAB.get(m["source_ontology"], m["source_ontology"])) for m in mappings
                 if m["source_id"] in (t["id"], t["id"] + "_all")]
        final += [(h.label, onto.upper()) for onto, hits in extra.items() for h in hits.get(t["label"], [])]
        final_hits = sorted({f"{lab} [{voc}]" for lab, voc in final if norm(lab) in gset})
        cand_hits = sorted({f"{c['label']} [{c['vocabulary']}]" for c in all_candidates[t["label"]] if norm(c["label"]) in gset})
        if not gset:        result_text = "no ground-truth term"
        elif final_hits:    result_text = "found as a final link"
        elif cand_hits:     result_text = "found among candidates only"
        else:               result_text = "not found"
        rows.append({"PANGAEA name": t["label"], "PANGAEA terms (ground truth)": " ; ".join(gt_terms), "result": result_text,
                     "found as final link": " ; ".join(final_hits), "found among candidates": " ; ".join(cand_hits),
                     "expert_verdict": ""})
    comparison = pd.DataFrame(rows)
    counts = comparison["result"].value_counts().reindex(
        ["found as a final link", "found among candidates only", "not found", "no ground-truth term"], fill_value=0).to_frame("number")
    counts["share"] = (counts["number"] / len(comparison)).map("{:.0%}".format)
    display(counts)
comparison


## 6. Save as Excel


In [ ]:
OUT = ROOT / "experiments" / "output_envo_eval" / "verdant_result.xlsx"
OUT.parent.mkdir(parents=True, exist_ok=True)
cands = [{"PANGAEA name": n, "vocabulary": c["vocabulary"], "candidate": c["label"], "uri": c["uri"]}
         for n, lst in all_candidates.items() for c in lst]
with pd.ExcelWriter(OUT) as xl:
    result.to_excel(xl, sheet_name="result", index=False)
    if len(comparison):
        comparison.to_excel(xl, sheet_name="comparison", index=False)
    pd.DataFrame(cands).to_excel(xl, sheet_name="all candidates", index=False)
print("Saved:", OUT)

## Findings

1. VERDANT was built for Natura 2000 habitats and species. For a habitat name it searches one vocabulary, ENVO.
   A PANGAEA parameter is a different kind of name. It often joins several parts, as in "Lactate, per fresh mass", which names both a chemical and a quantity.

2. VERDANT does not split a name into those parts. It searches the whole name, and also single words taken from it. For "gamma-Aminobutyric acid, per fresh mass" that brings back classes such as "acid rain" and "periglacial process", which only share a word with the parameter.

3. The pipeline itself does not search a chemical vocabulary. ChEBI was searched afterwards, as an extra step. That search treated the last word as the main one, so it returned "mass" for almost every chemical name.

4. On the 59 parameters of dataset PANGAEA.980734, 53 names have a PANGAEA term to compare with. VERDANT matched that term for 3 of them: Length, Width and Mass. The one ENVO link it kept was wrong: "Tissue type" became "tissue paper".

5. PANGAEA already splits these names and gives each part its own term. A linker for PANGAEA has to split the name first, then send each part to the vocabulary that holds it. That is what COMPASS is planned to do.
